# 🛒 Products Data Pipeline

**Description:** End-to-end pipeline that scrapes product data from [Books to Scrape](https://books.toscrape.com/), fetches additional products from a REST API, cleans and merges both datasets, then loads the result into a MySQL database.

| Step | Description |
|------|-------------|
| 1 | Web scraping with Selenium |
| 2 | REST API ingestion |
| 3 | Data cleaning & transformation |
| 4 | Export to CSV |
| 5 | Load into MySQL |

## 0 · Imports & Configuration

In [1]:
import requests
import pandas as pd
import mysql.connector as conn
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

In [2]:
# ── Configuration ────────────────────────────────────────────────────────────
SCRAPE_URL      = "https://books.toscrape.com/"
SCRAPE_LIMIT    = 10
API_URL         = "https://dummyjson.com/products"
OUTPUT_CSV      = "cleaned_data.csv"

DB_CONFIG = {
    "host":     "localhost",
    "user":     "root",
    "password": "pass",
    "database": "products_scrape_db",
}

## 1 · Web Scraping  —  Books to Scrape

In [13]:
def scrape_books(url: str, limit: int) -> pd.DataFrame:
    """Scrape `limit` book titles and prices from *Books to Scrape*.

    Returns a DataFrame with columns: title, price (str, raw £ value).
    """
    options = Options()
    options.add_argument("--headless")          # run without opening a browser window
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=options)
    records = []

    try:
        driver.get(url)
        items = driver.find_elements(By.CLASS_NAME, "product_pod")[:limit]

        for item in items:
            title = item.find_element(By.CSS_SELECTOR, "h3 a").get_attribute("title")
            price = item.find_element(By.CLASS_NAME, "price_color").text
            records.append({"title": title, "price": price})

        print(f"✅ Scraped {len(records)} books successfully.")
    except Exception as e:
        print(f"❌ Scraping failed: {e}")
    finally:
        driver.quit()

    return pd.DataFrame(records)

In [4]:
df_scraped = scrape_books(SCRAPE_URL, SCRAPE_LIMIT)
df_scraped.head()

✅ Scraped 10 books successfully.


,title,price
0,A Light in the Attic,£51.77
1,Tipping the Velvet,£53.74
2,Soumission,£50.10
3,Sharp Objects,£47.82
4,Sapiens: A Brief History of Humankind,£54.23


## 2 · API Ingestion  —  DummyJSON Products

In [5]:
def fetch_api_products(url: str) -> pd.DataFrame:
    """Fetch products from the DummyJSON REST API.

    Returns a DataFrame with columns: title, price.
    Falls back to an empty DataFrame on failure.
    """
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        products = response.json()["products"]
        df = pd.DataFrame(products)[["title", "price"]]
        print(f"✅ Fetched {len(df)} products from API.")
        return df
    except requests.exceptions.RequestException as e:
        print(f"❌ API request failed: {e}")
    except (KeyError, ValueError) as e:
        print(f"❌ Failed to parse API response: {e}")

    return pd.DataFrame(columns=["title", "price"])

In [6]:
df_api = fetch_api_products(API_URL)
df_api.head()

✅ Fetched 30 products from API.


,title,price
0,Essence Mascara Lash Princess,9.99
1,Eyeshadow Palette with Mirror,19.99
2,Powder Canister,14.99
3,Red Lipstick,12.99
4,Red Nail Polish,8.99


## 3 · Data Cleaning & Transformation

In [7]:
def clean_and_merge(df_scraped: pd.DataFrame, df_api: pd.DataFrame) -> pd.DataFrame:
    """Normalise, merge, and deduplicate the two source DataFrames.

    Steps
    -----
    1. Strip the £ symbol from scraped prices and cast to float.
    2. Ensure API prices are float.
    3. Concatenate both DataFrames.
    4. Strip whitespace from titles.
    5. Drop duplicate titles.
    6. Fill missing prices with 0.
    """
    # — Scraped: remove currency symbol
    df_scraped = df_scraped.copy()
    df_scraped["price"] = (
        df_scraped["price"]
        .str.replace("£", "", regex=False)
        .astype(float)
    )

    # — API: ensure numeric price
    df_api = df_api.copy()
    df_api["price"] = pd.to_numeric(df_api["price"], errors="coerce")

    # — Merge
    df = pd.concat([df_scraped, df_api], ignore_index=True)

    # — Clean
    df["title"] = df["title"].str.strip()
    df = df.drop_duplicates(subset="title")
    df["price"] = df["price"].fillna(0)

    print(f"✅ Cleaning complete — {len(df)} records retained.")
    return df.reset_index(drop=True)

In [8]:
final_df = clean_and_merge(df_scraped, df_api)

print(f"\nShape   : {final_df.shape}")
print(f"Dtypes  :\n{final_df.dtypes}")
print(f"Nulls   :\n{final_df.isnull().sum()}")
final_df.head(10)

✅ Cleaning complete — 40 records retained.

Shape   : (40, 2)
Dtypes  :
title     object
price    float64
dtype: object
Nulls   :
title    0
price    0
dtype: int64


,title,price
0,A Light in the Attic,51.77
1,Tipping the Velvet,53.74
2,Soumission,50.10
3,Sharp Objects,47.82
4,Sapiens: A Brief History of Humankind,54.23
5,The Requiem Red,22.65
6,The Dirty Little Secrets of Getting Your Dream...,33.34
7,The Coming Woman: A Novel Based on the Life of...,17.93
8,The Boys in the Boat: Nine Americans and Their...,22.60
9,The Black Maria,52.15


## 4 · Export to CSV

In [9]:
final_df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Data saved to '{OUTPUT_CSV}'")

✅ Data saved to 'cleaned_data.csv'


## 5 · Load into MySQL

In [10]:
def load_to_mysql(df: pd.DataFrame, config: dict) -> None:
    """Create the `products` table (if absent) and bulk-insert all rows.

    Parameters
    ----------
    df     : Cleaned DataFrame to persist.
    config : mysql.connector connection kwargs.
    """
    CREATE_TABLE = """
        CREATE TABLE IF NOT EXISTS products (
            id    INT AUTO_INCREMENT PRIMARY KEY,
            title VARCHAR(255) NOT NULL,
            price FLOAT        NOT NULL
        )
    """
    INSERT_ROW = "INSERT INTO products (title, price) VALUES (%s, %s)"

    try:
        db     = conn.connect(**config)
        cursor = db.cursor()

        cursor.execute(CREATE_TABLE)

        rows = [(row["title"], row["price"]) for _, row in df.iterrows()]
        cursor.executemany(INSERT_ROW, rows)    # single round-trip, much faster than a loop
        db.commit()

        print(f"✅ {cursor.rowcount} rows inserted into `products`.")

    except conn.Error as e:
        print(f"❌ MySQL error: {e}")
    finally:
        if 'cursor' in dir(): cursor.close()
        if 'db'     in dir(): db.close()

In [11]:
load_to_mysql(final_df, DB_CONFIG)

✅ 40 rows inserted into `products`.


## 6 · Quick Summary

In [12]:
print("Pipeline Summary")
print("=" * 40)
print(f"  Books scraped    : {len(df_scraped)}")
print(f"  API products     : {len(df_api)}")
print(f"  After dedup      : {len(final_df)}")
print(f"  CSV output       : {OUTPUT_CSV}")
print(f"  Price range      : £{final_df['price'].min():.2f} – £{final_df['price'].max():.2f}")
print("=" * 40)

final_df.describe()

Pipeline Summary
  Books scraped    : 10
  API products     : 30
  After dedup      : 40
  CSV output       : cleaned_data.csv
  Price range      : £0.99 – £2499.99


,price
count,40.000000
mean,174.595750
std,498.192092
min,0.990000
25%,8.490000
50%,18.960000
75%,53.862500
max,2499.990000
